**Part 2 · LangChain** · step 15 of 23  
[← 14 Human-in-the-loop](14_human_in_the_loop.ipynb) · [🏠 Roadmap](../README.md) · [16 First graph →](../03_LangGraph/16_first_graph.ipynb)

# 15 · Writing Your Own Middleware

Subclass `AgentMiddleware` and override any hook:

| Hook | Runs | Typical use |
|---|---|---|
| `before_agent(state, runtime)` | once, at the start of `invoke` | load user data, validate input |
| `before_model(state, runtime)` | before **every** LLM call | logging, trimming messages, guardrails |
| `after_model(state, runtime)` | after **every** LLM call | inspect / rewrite the reply, checks |
| `after_agent(state, runtime)` | once, at the end | save results, analytics |
| `wrap_model_call` / `wrap_tool_call` | around the call | retries, fallbacks, caching |

**Return value:** `None` = change nothing; a `dict` = state update (e.g. `{"messages": [...]}`).

There is also a decorator style: `@before_model`, `@after_model`, … for one-function middleware.

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.messages import AIMessage, HumanMessage, SystemMessage


class LoggingMiddleware(AgentMiddleware):
    """Print every incoming request before it reaches the model."""

    def before_model(self, state, runtime):
        print("Incoming request:", state["messages"][-1].content)
        return None  # observe only, no state change


class PrintReplyMiddleware(AgentMiddleware):
    """Print the model's reply after each model call (read-only)."""

    def after_model(self, state, runtime):
        print("Model replied:", state["messages"][-1].content[:80], "...")
        return None


class ReplaceReplyMiddleware(AgentMiddleware):
    """Overwrite the model's reply — shows that middleware can *change* state.

    Returning an AIMessage with the same ``id`` makes the ``add_messages`` reducer
    replace the original message instead of appending a new one.
    """

    def after_model(self, state, runtime):
        last = state["messages"][-1]
        return {"messages": [AIMessage(content="This response was replaced.", id=last.id)]}

In [ ]:
messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a 4 line poem on artificial intelligence"),
]

observer = create_agent(model="gpt-4o", middleware=[LoggingMiddleware(), PrintReplyMiddleware()])
result = observer.invoke({"messages": messages})

In [ ]:
replacer = create_agent(model="gpt-4o", middleware=[ReplaceReplyMiddleware()])
replacer.invoke({"messages": messages})["messages"][-1].content

## Decorator style

In [ ]:
from langchain.agents.middleware import before_model


@before_model
def log_message_count(state, runtime):
    """Print how many messages the model is about to receive."""
    print(f"Sending {len(state['messages'])} messages to the model")
    return None


agent = create_agent(model="gpt-4o-mini", middleware=[log_message_count])
agent.invoke({"messages": [HumanMessage("Hi!")]})["messages"][-1].content

## Other built-in middleware worth knowing
All importable from `langchain.agents.middleware`:

| Middleware | What it does |
|---|---|
| `ModelCallLimitMiddleware` / `ToolCallLimitMiddleware` | stop runaway loops by capping calls |
| `ModelFallbackMiddleware` | switch to a backup model when the primary fails |
| `ModelRetryMiddleware` / `ToolRetryMiddleware` | retry failed calls |
| `TodoListMiddleware` | gives the agent a planning / to-do tool |
| `LLMToolSelectorMiddleware` | pre-select relevant tools when there are many |
| `ContextEditingMiddleware` | clear old tool outputs to save context |

---
✅ **Done with 15 Custom middleware.** Next: [16 First graph →](../03_LangGraph/16_first_graph.ipynb)  
[🏠 Roadmap](../README.md)